# Day 3 — Data Structures and File I/O
## Master's Course: Quantitative Skills for Biology

Real biological data comes in tables, lists, mappings, and text files. Today we learn the Python data structures that hold this data and how to read/write files.

By the end of this session you will be able to:
- Choose the right data structure for a biological problem
- Use **lists**, **dictionaries**, **sets**, and **tuples** effectively
- Read and write text files in Colab
- Parse a FASTA file using pure Python

---
## 1. Lists

A **list** is an ordered, mutable sequence. It's the most general container in Python — use it when order matters or when you don't know the keys in advance.

In [ ]:
# Create a list of gene names
genes = ["BRCA1", "TP53", "EGFR", "MYC", "PTEN"]

# Indexing — Python is 0-based
print(genes[0])    # first element
print(genes[-1])   # last element
print(genes[1:3])  # slice: elements at index 1 and 2

In [ ]:
# Modifying a list
genes.append("BCL2")         # add to end
genes.insert(0, "KRAS")      # insert at position 0
genes.remove("MYC")          # remove by value
popped = genes.pop()         # remove and return last element

print(genes)
print("Removed:", popped)

In [ ]:
# Useful list functions
print("Length:", len(genes))
print("Sorted:", sorted(genes))
print("Index of TP53:", genes.index("TP53"))
print("Is EGFR in list?", "EGFR" in genes)

# Nested lists — e.g., [gene_name, expression_value]
expression_data = [
    ["BRCA1", 8.43],
    ["TP53",  7.12],
    ["EGFR",  9.01]
]
for row in expression_data:
    print(f"  {row[0]}: {row[1]:.2f}")

---
## 2. Dictionaries

A **dictionary** maps **keys** to **values**. It is the natural structure for gene→expression mappings, sample metadata, and any labelled data.

Lookup in a dict is very fast regardless of size — much better than scanning a list.

In [ ]:
# Create a dict: gene name → expression value
expression = {
    "BRCA1": 8.43,
    "TP53":  7.12,
    "EGFR":  9.01,
    "MYC":   11.5,
    "PTEN":  5.30
}

# Access a value by key
print(expression["BRCA1"])

# Safe access — returns None if key doesn't exist
print(expression.get("UNKNOWN_GENE"))         # None
print(expression.get("UNKNOWN_GENE", 0.0))    # default value

In [ ]:
# Iterate over a dictionary
for gene, value in expression.items():
    print(f"  {gene}: {value:.2f}")

print("\nAll gene names:", list(expression.keys()))
print("All values:", list(expression.values()))

In [ ]:
# Add or update entries
expression["KRAS"] = 6.77    # add new key
expression["TP53"] = 7.55    # update existing key

print("Updated TP53:", expression["TP53"])
print("Total genes:", len(expression))

In [ ]:
# Dict comprehension — build from two lists
gene_names = ["BRCA1", "TP53", "EGFR"]
values     = [8.43,    7.12,   9.01]

expr_dict = {gene: val for gene, val in zip(gene_names, values)}
print(expr_dict)

---
## 3. Sets

A **set** is an unordered collection of **unique** values. Sets are ideal for overlap and difference operations between gene lists — a very common bioinformatics task.

In [ ]:
# Genes detected in two different experiments
experiment_A = {"BRCA1", "TP53", "EGFR", "MYC", "PTEN"}
experiment_B = {"TP53", "KRAS", "EGFR", "BCL2", "RB1"}

# Intersection — genes in BOTH experiments
shared = experiment_A & experiment_B
print("Shared genes:", shared)

# Union — genes in EITHER experiment
all_genes = experiment_A | experiment_B
print("All genes:", all_genes)

# Difference — genes unique to experiment A
only_A = experiment_A - experiment_B
print("Only in A:", only_A)

# Symmetric difference — genes in one but not both
exclusive = experiment_A ^ experiment_B
print("Exclusive:", exclusive)

In [ ]:
# Sets automatically remove duplicates
raw_list = ["BRCA1", "TP53", "BRCA1", "MYC", "TP53", "EGFR"]
unique_genes = set(raw_list)
print("Unique genes:", unique_genes)
print("Duplicates removed:", len(raw_list) - len(unique_genes))

---
## 4. Tuples

A **tuple** is like a list but **immutable** — once created, it cannot be changed. Use tuples for data that should not change, like genomic coordinates.

In [ ]:
# Genomic coordinates: (chromosome, start, end)
brca1_locus = ("chr17", 43044295, 43125483)
print("Chromosome:", brca1_locus[0])
print("Start:",      brca1_locus[1])
print("End:",        brca1_locus[2])
print("Length:",     brca1_locus[2] - brca1_locus[1])

# Tuple unpacking — assign each element to a variable
chrom, start, end = brca1_locus
print(f"{chrom}:{start}-{end}")

### Choosing the right structure

| Structure | Ordered | Mutable | Duplicates | Best for |
|-----------|---------|---------|------------|----------|
| `list` | Yes | Yes | Yes | Sequences of items |
| `dict` | Yes* | Yes | Keys: No | Key→value mapping |
| `set` | No | Yes | No | Membership, overlap |
| `tuple` | Yes | No | Yes | Fixed records (coordinates) |

*Dicts preserve insertion order in Python 3.7+

---
## 5. Writing and reading files in Colab

Google Colab has a local filesystem for the duration of your session. Files written here disappear when the session ends, but you can download them.

In [ ]:
# Write a TSV file (tab-separated values)
gene_data = [
    ("BRCA1", 8.43),
    ("TP53",  7.12),
    ("EGFR",  9.01),
    ("MYC",   11.5),
    ("PTEN",  5.30)
]

with open("expression.tsv", "w") as f:
    f.write("gene\texpression\n")   # header line
    for gene, value in gene_data:
        f.write(f"{gene}\t{value}\n")

print("File written.")

In [ ]:
# Read it back line by line
expression_dict = {}

with open("expression.tsv", "r") as f:
    header = f.readline()   # skip the header
    for line in f:
        line = line.strip()         # remove trailing newline
        parts = line.split("\t")    # split on tab
        gene = parts[0]
        value = float(parts[1])
        expression_dict[gene] = value

print("Loaded:", expression_dict)

---
## 6. Parsing a FASTA file

FASTA is the most common format for biological sequences. Each entry has:
- A **header line** starting with `>`
- One or more **sequence lines** following it

Let's write a parser from scratch.

In [ ]:
# First, create a sample FASTA file
fasta_content = """>BRCA1_exon1
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAAATCTTA
GAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGACCACATATTTTGC
>TP53_CDS
ATGGAGGAGCCGCAGTCGGATCCGACCTGCCTCTGCTTCACCTGGTGGATGACCCTGAACCCTGCT
ATGGCCCTGGCTGCAGCACCAGTGGCCCAGGGCACCCTGGGGCCTGAGGCCAGCCTGGGCTCCGGGG
>EGFR_kinase
ATGCGACCCTCCGGGACGGCCGGGGCAGCGCTCCTGGCGCTGCTGGCTGCGCTCTGCCCGGCGAGTC
"""

with open("sequences.fasta", "w") as f:
    f.write(fasta_content)

print("FASTA file created.")

In [ ]:
def parse_fasta(filename):
    """
    Parse a FASTA file and return a dict {header: sequence}.
    Multi-line sequences are concatenated.
    """
    sequences = {}
    current_header = None
    current_seq = []

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line:              # skip blank lines
                continue
            if line.startswith(">"):
                # Save the previous sequence
                if current_header is not None:
                    sequences[current_header] = "".join(current_seq)
                # Start a new sequence
                current_header = line[1:]   # remove the '>'
                current_seq = []
            else:
                current_seq.append(line)    # accumulate sequence lines

    # Don't forget the last sequence
    if current_header is not None:
        sequences[current_header] = "".join(current_seq)

    return sequences

# Parse the file
seqs = parse_fasta("sequences.fasta")

for header, seq in seqs.items():
    print(f">{header}")
    print(f"  Length: {len(seq)} bp")
    print(f"  First 20 bases: {seq[:20]}")
    print()

---
## Exercises

### Exercise 1
The string below represents the contents of a two-column TSV file (gene name and read count).

Parse it into a dictionary `{gene: count}`, then print all genes where the count is greater than 100.

In [ ]:
tsv_data = """gene\tcount
BRCA1\t245
TP53\t88
EGFR\t312
MYC\t150
PTEN\t43
KRAS\t198
BCL2\t67
RB1\t201
AKT1\t55
CDK4\t130"""

# Exercise 1 — your code here
# Hint: tsv_data.split("\n") splits into lines


### Exercise 2
Given two lists of gene IDs from two different studies:

```python
study1 = ["BRCA1", "TP53", "EGFR", "MYC", "PTEN", "AKT1"]
study2 = ["TP53", "KRAS", "EGFR", "BCL2", "RB1", "BRCA2", "AKT1"]
```

Using sets, find:
1. Genes present in **both** studies
2. Genes **unique to study 1**
3. Genes **unique to study 2**
4. Total number of unique genes across both studies

In [ ]:
# Exercise 2 — your code here
study1 = ["BRCA1", "TP53", "EGFR", "MYC", "PTEN", "AKT1"]
study2 = ["TP53", "KRAS", "EGFR", "BCL2", "RB1", "BRCA2", "AKT1"]


### Exercise 3
Extend the `parse_fasta()` function to also return the **length** of each sequence.

Modify it so it returns a dict like: `{header: {"sequence": seq, "length": len(seq)}}`

Test it on `sequences.fasta` and print the name and length of each entry.

In [ ]:
# Exercise 3 — your code here
def parse_fasta_with_length(filename):
    pass  # replace this

# Test:
# result = parse_fasta_with_length("sequences.fasta")
# for header, info in result.items():
#     print(f"{header}: {info['length']} bp")


---
## Summary

| Structure | When to use |
|-----------|------------|
| `list` | Ordered collection — gene lists, sample names |
| `dict` | Labelled data — gene→value mappings |
| `set` | Unique items, overlap between gene sets |
| `tuple` | Fixed records — genomic coordinates |

| File task | Code pattern |
|-----------|-------------|
| Write | `with open(file, 'w') as f: f.write(...)` |
| Read | `with open(file, 'r') as f: for line in f:` |
| Parse FASTA | Detect `>` headers, accumulate sequence lines |

**Next session:** NumPy and pandas — the core tools for numerical and tabular data analysis.